In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HTML, Output, Layout, GridBox
from IPython.display import display, clear_output

# ============================================================
# CHARACTERISTIC FUNCTIONS AND THE QUANTIZATION THEOREM
# ============================================================

# ============================================================
# CSS
# ============================================================

display(HTML("""
<style>

.quant-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 22px !important;
}

.quant-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}

.quant-radio > label {
    display: none !important;
}

</style>
"""))

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:1080px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#315b8a;
    margin-bottom:8px;
">
Characteristic Functions and the Quantization Theorem
</div>

<div style="margin-bottom:4px;">
Quantization produces periodic replicas of the input characteristic function, separated by the radial quantization frequency ξ = 2π/Δ.
</div>

<div style="margin-bottom:4px;">
For a bandlimited characteristic function with support |u| ≤ B, the replicas do not overlap when B ≤ π/Δ.
</div>

<div style="margin-bottom:4px;">
This is the quantization-domain analogue of the Nyquist sampling condition: decreasing Δ increases ξ and separates the replicas.
</div>

<div>
<b>This notebook:</b> compares the exact bandlimited case with a Gaussian characteristic function, which is not strictly bandlimited.
</div>

</div>
""")

# ============================================================
# INPUT MODEL SELECTOR
# ============================================================

model_selector = RadioButtons(
    options=['Bandlimited', 'Gaussian'],
    value='Bandlimited',
    description='',
    layout=Layout(width='250px')
)

model_selector.add_class('quant-radio')

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(
    width='155px'
)

delta_slider = FloatSlider(
    min=0.40,
    max=2.00,
    step=0.05,
    value=1.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

B_slider = FloatSlider(
    min=0.50,
    max=5.00,
    step=0.25,
    value=2.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

sigma_slider = FloatSlider(
    min=0.40,
    max=2.00,
    step=0.10,
    value=1.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

copies_slider = IntSlider(
    min=1,
    max=5,
    step=1,
    value=3,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

delta_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>'
)

B_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">2.00</div>'
)

sigma_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>'
)

copies_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">3</div>'
)

# ============================================================
# CONTROL LABELS
# ============================================================

model_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Input model:</div>'
)

delta_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Quantization step Δ:</div>'
)

B_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Half-bandwidth B:</div>'
)

sigma_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Gaussian σ:</div>'
)

copies_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Replica pairs:</div>'
)

empty_value = HTML(
    '<div></div>'
)

# ============================================================
# CONTROL GRID
# ============================================================

controls_grid = GridBox(
    children=[
        model_label, model_selector, empty_value,
        delta_label, delta_slider, delta_value,
        B_label, B_slider, B_value,
        sigma_label, sigma_slider, sigma_value,
        copies_label, copies_slider, copies_value
    ],
    layout=Layout(
        width='930px',
        grid_template_columns='145px 255px 55px 145px 155px 55px',
        grid_template_rows='34px 34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#315b8a;
            margin-bottom:5px;
        ">
        Quantization-Domain Parameters
        </div>
        """),

        controls_grid
    ],
    layout=Layout(
        width='960px',
        padding='10px 14px',
        border='1px solid #bfd0e2',
        margin='10px 0px 8px 0px',
        overflow='hidden'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# GRAPH OUTPUT
# ============================================================

graph_output = Output(
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

# ============================================================
# CHARACTERISTIC FUNCTION
# ============================================================

def characteristic_function(u, model, B, sigma):

    if model == 'Bandlimited':

        phi = np.maximum(
            1.0 - np.abs(u) / B,
            0.0
        )

    else:

        phi = np.exp(
            -0.5 * sigma**2 * u**2
        )

    return phi

# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_quantization_theorem(model, delta, B, sigma, replica_pairs):

    # --------------------------------------------------------
    # CHARACTERISTIC-FUNCTION AXIS
    # --------------------------------------------------------

    u = np.linspace(
        -20.0,
        20.0,
        4000
    )

    # --------------------------------------------------------
    # RADIAL QUANTIZATION FREQUENCY
    #
    # xi = 2*pi/Delta
    # --------------------------------------------------------

    xi = 2.0 * np.pi / delta

    nyquist_limit = np.pi / delta

    # --------------------------------------------------------
    # INPUT CHARACTERISTIC FUNCTION
    # --------------------------------------------------------

    phi_x = characteristic_function(
        u,
        model,
        B,
        sigma
    )

    # --------------------------------------------------------
    # OUTPUT CHARACTERISTIC FUNCTION
    #
    # Phi_y(u) =
    #
    # Sum_l Phi_x(u + l*xi)
    #       sinc[Delta(u + l*xi)/2]
    #
    # np.sinc(q) = sin(pi*q)/(pi*q)
    # --------------------------------------------------------

    phi_y = np.zeros_like(
        u
    )

    for ell in range(
        -replica_pairs,
        replica_pairs + 1
    ):

        shifted_argument = u + ell * xi

        replica = characteristic_function(
            shifted_argument,
            model,
            B,
            sigma
        )

        sinc_factor = np.sinc(
            delta * shifted_argument / (2.0 * np.pi)
        )

        phi_y += replica * sinc_factor

    # --------------------------------------------------------
    # CENTRAL REPLICA
    # --------------------------------------------------------

    central_term = phi_x * np.sinc(
        delta * u / (2.0 * np.pi)
    )

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(10.5, 7.0)
    )

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[1.0, 1.05],
        hspace=0.43,
        wspace=0.30
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax3 = fig.add_subplot(
        gs[1, :]
    )

    # ========================================================
    # GRAPH 1:
    # INPUT CHARACTERISTIC FUNCTION
    # ========================================================

    ax1.plot(
        u,
        phi_x,
        linewidth=2.0
    )

    if model == 'Bandlimited':

        ax1.axvline(
            -B,
            linestyle='--',
            linewidth=1.0
        )

        ax1.axvline(
            B,
            linestyle='--',
            linewidth=1.0
        )

    ax1.axvline(
        -nyquist_limit,
        linestyle=':',
        linewidth=1.2,
        label='±π/Δ'
    )

    ax1.axvline(
        nyquist_limit,
        linestyle=':',
        linewidth=1.2
    )

    ax1.set_xlim(
        -10,
        10
    )

    ax1.set_ylim(
        -0.10,
        1.15
    )

    ax1.set_xlabel(
        'Characteristic-function variable u',
        fontsize=10
    )

    ax1.set_ylabel(
        'Φₓ(u)',
        fontsize=10
    )

    ax1.set_title(
        'Input Characteristic Function',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax1.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # GRAPH 2:
    # PERIODIC REPLICAS
    # ========================================================

    for ell in range(
        -replica_pairs,
        replica_pairs + 1
    ):

        shifted_argument = u + ell * xi

        replica = characteristic_function(
            shifted_argument,
            model,
            B,
            sigma
        )

        ax2.plot(
            u,
            replica,
            linewidth=1.2,
            alpha=0.75
        )

    ax2.set_xlim(
        -20,
        20
    )

    ax2.set_ylim(
        -0.10,
        1.15
    )

    ax2.set_xlabel(
        'Characteristic-function variable u',
        fontsize=10
    )

    ax2.set_ylabel(
        'Shifted copies of Φₓ(u)',
        fontsize=10
    )

    ax2.set_title(
        'Replicas Spaced by ξ = 2π/Δ',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    # ========================================================
    # GRAPH 3:
    # OUTPUT CHARACTERISTIC FUNCTION
    # ========================================================

    ax3.plot(
        u,
        phi_y,
        linewidth=2.0,
        label='Φᵧ(u)'
    )

    ax3.plot(
        u,
        central_term,
        linestyle='--',
        linewidth=1.7,
        label='Central replica'
    )

    for ell in range(
        -replica_pairs,
        replica_pairs + 1
    ):

        if ell != 0:

            ax3.axvline(
                -ell * xi,
                linestyle=':',
                linewidth=0.7,
                alpha=0.4
            )

    ax3.axhline(
        0,
        linewidth=0.8
    )

    ax3.set_xlim(
        -20,
        20
    )

    ax3.set_ylim(
        -0.35,
        1.15
    )

    ax3.set_xlabel(
        'Characteristic-function variable u',
        fontsize=10
    )

    ax3.set_ylabel(
        'Characteristic function',
        fontsize=10
    )

    ax3.set_title(
        'Characteristic Function after Quantization',
        fontsize=12,
        pad=8
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax3.legend(
        loc='upper right',
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.93,
        bottom=0.09
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # QUANTIZATION-THEOREM STATUS
    # ========================================================

    if model == 'Bandlimited':

        if B <= nyquist_limit:

            theorem_status = """
            <span style="
                color:#1d7b38;
                font-weight:bold;
            ">
            SATISFIED — the replicas do not overlap.
            </span>
            """

        else:

            theorem_status = """
            <span style="
                color:#b83232;
                font-weight:bold;
            ">
            NOT SATISFIED — the replicas overlap.
            </span>
            """

        condition_text = (
            f'B = {B:.3f}'
            f' &nbsp;&nbsp; ≤? &nbsp;&nbsp; '
            f'π/Δ = {nyquist_limit:.3f}'
        )

    else:

        theorem_status = """
        <span style="
            color:#b36b00;
            font-weight:bold;
        ">
        NOT STRICTLY SATISFIED — the Gaussian characteristic
        function is not bandlimited.
        </span>
        """

        condition_text = (
            'Gaussian Φₓ(u) has nonzero tails '
            'for every finite u.'
        )

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.48;
        width:960px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Quantization step:</b>
    Δ = {delta:.3f}

    &nbsp;&nbsp;&nbsp;

    <b>Radial quantization frequency:</b>
    ξ = 2π/Δ = {xi:.3f}

    <br>

    <b>First quantization theorem:</b>
    {condition_text}

    <br>

    {theorem_status}

    </div>
    """

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    # --------------------------------------------------------
    # ENABLE / DISABLE MODEL PARAMETERS
    # --------------------------------------------------------

    if model_selector.value == 'Bandlimited':

        B_slider.disabled = False
        sigma_slider.disabled = True

    else:

        B_slider.disabled = True
        sigma_slider.disabled = False

    # --------------------------------------------------------
    # UPDATE NUMERICAL LABELS
    # --------------------------------------------------------

    delta_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {delta_slider.value:.2f}
    </div>
    """

    B_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {B_slider.value:.2f}
    </div>
    """

    sigma_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {sigma_slider.value:.2f}
    </div>
    """

    copies_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {copies_slider.value}
    </div>
    """

    # --------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------

    with graph_output:

        clear_output(
            wait=True
        )

        plot_quantization_theorem(
            model_selector.value,
            delta_slider.value,
            B_slider.value,
            sigma_slider.value,
            copies_slider.value
        )

# ============================================================
# CONNECT EVERY CONTROL DIRECTLY TO UPDATE FUNCTION
# ============================================================

model_selector.observe(
    update_notebook,
    names='value'
)

delta_slider.observe(
    update_notebook,
    names='value'
)

B_slider.observe(
    update_notebook,
    names='value'
)

sigma_slider.observe(
    update_notebook,
    names='value'
)

copies_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1080px;
    padding:11px 15px;
    border:1px solid #c5d5e5;
    background:#f7fbff;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#315b8a;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The quantization step Δ determines the spacing ξ = 2π/Δ between replicas of the input characteristic function.
</div>

<div style="margin-bottom:4px;">
Decreasing Δ moves the replicas farther apart. For a bandlimited Φₓ(u), no overlap occurs when its half-bandwidth satisfies B ≤ π/Δ.
</div>

<div>
This is directly analogous to avoiding spectral overlap in ordinary sampling; the first quantization theorem is therefore the quantization-domain counterpart of the Nyquist sampling theorem.
</div>

</div>
""")

# ============================================================
# INITIAL DRAW
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        controls_card,
        graph_output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

display(main_layout)